In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ================================
# 🌳 Optimized Decision Tree – Devign Dataset
# ================================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ================================
# 🔹 1. LOAD DATASET
# ================================
data_path = "/kaggle/input/datasets/nikunjnawal009/decision-devignx"
files = os.listdir(data_path)
print("Files in dataset:", files)

# Load first CSV file
file_path = os.path.join(data_path, files[0])
df = pd.read_csv(file_path)

print("\nColumns:", df.columns.tolist())
print("\nSample Data:\n", df.head())

# ================================
# 🔹 2. FIX COLUMN NAMES
# ================================
col_mapping = {
    "function": "code", "func": "code", "code_snippet": "code",
    "target": "label", "vul": "label"
}
df.rename(columns={k: v for k, v in col_mapping.items() if k in df.columns}, inplace=True)
df = df[["code", "label"]].dropna()
df["label"] = df["label"].astype(int)

print(f"\nDataset size: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# ================================
# 🔹 3. TRAIN-TEST SPLIT (Stratified)
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    df["code"], df["label"],
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

# ================================
# 🔹 4. TF-IDF FEATURE EXTRACTION (More features)
# ================================
vectorizer = TfidfVectorizer(
    max_features=20000,          # increased vocabulary
    ngram_range=(1, 2),          # word unigrams + bigrams
    sublinear_tf=True            # often improves performance
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

# ================================
# 🔹 5. HYPERPARAMETER TUNING (with validation set)
# ================================
# Split training data again for validation
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train_tfidf, y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

param_grid = {
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'class_weight': ['balanced', None]
}

dt = DecisionTreeClassifier(random_state=42)
grid = GridSearchCV(
    dt, param_grid,
    scoring='f1',                # optimize F1 score
    cv=3,                        # internal 3-fold CV on training data
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train2, y_train2)
print("\n✅ Best parameters found:", grid.best_params_)

# ================================
# 🔹 6. EVALUATE ON VALIDATION SET
# ================================
val_preds = grid.predict(X_val)
print("\n🔍 Validation metrics before final test:")
print(f"Accuracy : {accuracy_score(y_val, val_preds):.4f}")
print(f"Precision: {precision_score(y_val, val_preds):.4f}")
print(f"Recall   : {recall_score(y_val, val_preds):.4f}")
print(f"F1 Score : {f1_score(y_val, val_preds):.4f}")

# ================================
# 🔹 7. FINAL MODEL & TEST EVALUATION
# ================================
best_model = grid.best_estimator_
best_model.fit(X_train_tfidf, y_train)
y_pred = best_model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n📊 FINAL TEST RESULTS")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print("\n🔍 Classification Report:\n")
print(classification_report(y_test, y_pred))

Files in dataset: ['Devignx_validation.csv', 'devignx_test.csv', 'devignx_train.csv']

Columns: ['code', 'label']

Sample Data:
                                                 code  label
0  static int V A R1 F U N1 ( VAR1 * V A R2 ) { V...      1
1  static int F U N1 ( VAR1 * V A R2 , const char...      0
2  static int F U N1 ( VAR1 * V A R2 ) { int V A ...      0
3  static VAR1 F U N1 ( VAR1 * V A R2 , void * V ...      0
4  static inline void F U N1 ( VAR1 ) ( const VAR...      1

Dataset size: 2732
Label distribution:
label
0    1545
1    1187
Name: count, dtype: int64
TF-IDF matrix shape: (2458, 20000)
Fitting 3 folds for each of 90 candidates, totalling 270 fits

✅ Best parameters found: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2}

🔍 Validation metrics before final test:
Accuracy : 0.4580
Precision: 0.4371
Recall   : 0.8688
F1 Score : 0.5816

📊 FINAL TEST RESULTS
Accuracy : 0.5584
Precision: 0.4722
Recall   : 0.1429
F1 Score : 0.2